## Mapeando as Condições de Exibição

In [1]:
import json
import os
import unicodedata
from collections import OrderedDict
from glob import glob
from pathlib import Path

# ==============================
# 1. Configuracao e validacoes
# ==============================

CONFIG_PRIVATE_PATH = Path("../config/arquivos_complementares.private.json")
CONFIG_EXAMPLE_PATH = Path("../config/arquivos_complementares.example.json")

if CONFIG_PRIVATE_PATH.exists():
    _cfg = json.loads(CONFIG_PRIVATE_PATH.read_text(encoding="utf-8"))
    print("Configuracao privada carregada.")
elif CONFIG_EXAMPLE_PATH.exists():
    _cfg = json.loads(CONFIG_EXAMPLE_PATH.read_text(encoding="utf-8"))
    print("Configuracao example carregada (use private para ambiente local).")
else:
    raise FileNotFoundError(
        "Nenhum arquivo de configuracao encontrado em ../config/arquivos_complementares.*.json"
    )

QUADROS_GLOB = _cfg.get("quadros_glob")
MAPA_VARIAVEIS_PATH = _cfg.get("mapa_variaveis_path")
OUTPUT_CONDICOES_PATH = _cfg.get("output_condicoes_path")
QUESTION_BLOCK_KEYS = _cfg.get(
    "question_block_keys",
    ["Quesitos", "QuesitoProdutos", "ListaProdutos", "QuesitoProdutosAgrupados", "ListaDeProdutos"],
)
CONDITION_NORMALIZATION = _cfg.get(
    "condition_normalization",
    {"&&": "and", "||": "or", "null": "None", "X_": ""},
)
LOGGING = _cfg.get("logging", {})

assert QUADROS_GLOB, "Configure 'quadros_glob' no arquivo de configuracao."
assert MAPA_VARIAVEIS_PATH, "Configure 'mapa_variaveis_path' no arquivo de configuracao."
assert OUTPUT_CONDICOES_PATH, "Configure 'output_condicoes_path' no arquivo de configuracao."

_QUADROS = sorted(glob(QUADROS_GLOB))
_MAPA_PATH = Path(MAPA_VARIAVEIS_PATH)
_OUT_COND = Path(OUTPUT_CONDICOES_PATH)

assert _QUADROS, f"Nenhum arquivo encontrado com quadros_glob={QUADROS_GLOB}"
assert _MAPA_PATH.exists(), f"Arquivo nao encontrado: {_MAPA_PATH}"
_OUT_COND.parent.mkdir(parents=True, exist_ok=True)

print(f"Quadros encontrados: {len(_QUADROS)}")
print(f"Mapa de variaveis: {_MAPA_PATH.resolve()}")
print(f"Saida condicoes: {_OUT_COND.resolve()}")


def normalize_condition(expr: str, replacements: dict[str, str]) -> str:
    text = str(expr or "")
    for src, dst in replacements.items():
        text = text.replace(src, dst)
    return text.strip()


def iter_question_blocks(quadro_data: dict, block_keys: list[str]):
    for key in block_keys:
        value = quadro_data.get(key, None)
        if value is None:
            continue
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    yield item
        elif isinstance(value, dict):
            yield value


def safe_questions(block: dict):
    perguntas = block.get("Perguntas", [])
    if isinstance(perguntas, list):
        return [p for p in perguntas if isinstance(p, dict)]
    return []


def merge_display_conditions(old_value: str, new_value: str) -> str:
    left = (old_value or "").strip()
    right = (new_value or "").strip()
    if not left or not right:
        return ""
    if left == right:
        return left
    return f"({left}) or ({right})"


# ==============================
# 2. Mapeando as condicoes de exibicao
# ==============================

ocorrencias = 0
cond_exib_quadros = {}
cond_exib_perguntas = {}

for q_path in _QUADROS:
    with open(q_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    quadro_nome = Path(q_path).stem
    quadro_cond = normalize_condition(data.get("CondicaoDeExibicao", ""), CONDITION_NORMALIZATION)
    cond_exib_quadros[quadro_nome] = quadro_cond

    for quesito in iter_question_blocks(data, QUESTION_BLOCK_KEYS):
        quesito_cond = normalize_condition(quesito.get("CondicaoDeExibicao", ""), CONDITION_NORMALIZATION)

        for pergunta in safe_questions(quesito):
            codigo_variavel = pergunta.get("CodigoVariavel")
            if not codigo_variavel:
                continue

            pergunta_cond = normalize_condition(
                pergunta.get("CondicaoDeExibicao", ""),
                CONDITION_NORMALIZATION,
            )

            parts = []
            if quesito_cond:
                parts.append(f"({quesito_cond})")
            if pergunta_cond:
                parts.append(f"({pergunta_cond})")
            variavel_cond = " and ".join(parts).strip()

            if codigo_variavel in cond_exib_perguntas:
                ocorrencias += 1
                antigo = cond_exib_perguntas[codigo_variavel]
                combinado = merge_display_conditions(antigo, variavel_cond)
                cond_exib_perguntas[codigo_variavel] = combinado
                if LOGGING.get("show_duplicates", False):
                    print(f"Variavel {codigo_variavel} encontrada novamente.")
                    print(f"Condição existente: {antigo}")
                    print(f"Nova condição: {variavel_cond}")
                    print(f"Condição combinada: {combinado}")
                    print("-" * 100)
            else:
                cond_exib_perguntas[codigo_variavel] = variavel_cond

cond_exibicao = {
    "quadros": cond_exib_quadros,
    "perguntas": cond_exib_perguntas,
}

with open(_OUT_COND, "w", encoding="utf-8") as f:
    json.dump(cond_exibicao, f, indent=4, ensure_ascii=False)

print(f"Total de variaveis com condicoes de exibicao: {len(cond_exib_perguntas)}")
print(f"Total de variaveis com condicao combinada: {ocorrencias}")
print(f"Arquivo salvo em: {_OUT_COND.resolve()}")

Configuracao privada carregada.
Quadros encontrados: 40
Mapa de variaveis: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/mapa_variaveis_new.json
Saida condicoes: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/generated/condicoes_exibicao.json
Total de variaveis com condicoes de exibicao: 538
Total de variaveis com condicao combinada: 19
Arquivo salvo em: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/generated/condicoes_exibicao.json


## Mapeando as Variáveis por Quadro

In [ ]:
# ==============================
# 3. Mapeando as variaveis por quadro
# ==============================

import re

OUTPUT_FORM_STRUCTURE_PATH = _cfg.get("output_form_structure_path")
REFERENCE_FORM_STRUCTURE_PATH = _cfg.get("reference_form_structure_path")
CLASSIFICATION_CFG = _cfg.get("classification", {})

assert OUTPUT_FORM_STRUCTURE_PATH, "Configure 'output_form_structure_path' no arquivo de configuracao."
assert REFERENCE_FORM_STRUCTURE_PATH, "Configure 'reference_form_structure_path' no arquivo de configuracao."

_OUT_FORM = Path(OUTPUT_FORM_STRUCTURE_PATH)
_REF_FORM = Path(REFERENCE_FORM_STRUCTURE_PATH)
_OUT_FORM.parent.mkdir(parents=True, exist_ok=True)

assert _REF_FORM.exists(), f"Arquivo de referencia nao encontrado: {_REF_FORM}"

with open(_MAPA_PATH, "r", encoding="utf-8") as f:
    variable_map = json.load(f)
with open(_REF_FORM, "r", encoding="utf-8") as f:
    reference_form = json.load(f)


def normalize_var_code(var_code: str) -> str:
    value = str(var_code or "").strip()
    if value.startswith("X_"):
        return value[2:]
    return value


def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text or ""))
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def normalize_text(text: str) -> str:
    value = str(text or "").lower().strip()
    if CLASSIFICATION_CFG.get("normalize_unicode", True):
        value = strip_accents(value)
    return " ".join(value.split())


def build_text_blob_from_dict(data: dict) -> str:
    parts = []
    for _, value in data.items():
        if isinstance(value, str):
            parts.append(value)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, str):
                    parts.append(item)
    return " ".join(parts)


def candidate_texts_from_variable_map(var_code: str) -> dict[str, str]:
    entry = variable_map.get(var_code, {})
    if not isinstance(entry, dict):
        return {"description": "", "reference": ""}

    description_parts = []
    reference_parts = []

    for key, value in entry.items():
        if not isinstance(value, str):
            continue
        key_n = normalize_text(key)
        if any(token in key_n for token in ["name", "nome", "description", "descricao", "label", "texto"]):
            description_parts.append(value)
        if any(token in key_n for token in ["reference", "referencia", "fonte", "origem"]):
            reference_parts.append(value)

    return {
        "description": " ".join(description_parts),
        "reference": " ".join(reference_parts),
    }


SECTION_KEYWORDS = CLASSIFICATION_CFG.get("section_keywords", {})
TEXT_FIELDS_PRIORITY = CLASSIFICATION_CFG.get(
    "text_fields_priority",
    ["description", "reference", "question_text", "question_group_text", "quadro_title"],
)
FALLBACK_SECTION = CLASSIFICATION_CFG.get("fallback_section", "information")
ENFORCE_REFERENCE_OUTPUT = CLASSIFICATION_CFG.get("enforce_reference_output", True)
INCLUDE_SHOW_UNITS = CLASSIFICATION_CFG.get("include_show_units", True)


def score_by_keywords(text: str, keywords: list[str]) -> int:
    t = normalize_text(text)
    if not t:
        return 0
    score = 0
    for kw in keywords:
        kw_n = normalize_text(kw)
        if kw_n and kw_n in t:
            score += 1
    return score


def classify_section(context: dict[str, str]) -> str:
    scores = {
        "information": 0,
        "area": 0,
        "production": 0,
        "sells": 0,
    }

    total_fields = len(TEXT_FIELDS_PRIORITY)
    field_weight = {field: total_fields - idx for idx, field in enumerate(TEXT_FIELDS_PRIORITY)}

    for field in TEXT_FIELDS_PRIORITY:
        text_value = context.get(field, "")
        weight = field_weight.get(field, 1)
        for section_name in scores:
            kw_list = SECTION_KEYWORDS.get(section_name, [])
            scores[section_name] += score_by_keywords(text_value, kw_list) * weight

    best_score = max(scores.values())
    if best_score <= 0:
        return FALLBACK_SECTION

    for section_name in ["sells", "production", "area", "information"]:
        if scores[section_name] == best_score:
            return section_name
    return FALLBACK_SECTION


def append_unique(target_list: list[str], value: str):
    if value and value not in target_list:
        target_list.append(value)


def iter_all_question_groups(quadro_data: dict):
    for key in QUESTION_BLOCK_KEYS:
        value = quadro_data.get(key, None)
        if value is None:
            continue
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    yield item
        elif isinstance(value, dict):
            yield value


def extract_var_codes_from_condition(expr: str) -> set[str]:
    raw = re.findall(r"\b(?:X_)?V[WX]?\d+\b", str(expr or ""))
    return {normalize_var_code(x) for x in raw}


def build_candidate_form_structure():
    quad_map = OrderedDict()

    for quadro_file in _QUADROS:
        with open(quadro_file, "r", encoding="utf-8") as f:
            quadro_data = json.load(f)

        quadro_name = Path(quadro_file).stem
        quadro_title = quadro_data.get("Descricao", "")

        item = {
            "title": quadro_title,
            "variables": [],
            "sections": {
                "information": [],
                "area": [],
                "production": [],
                "sells": [],
            },
        }

        for derived_var in quadro_data.get("VariaveisDerivadas", []):
            append_unique(item["variables"], normalize_var_code(derived_var))

        for group in iter_all_question_groups(quadro_data):
            group_text = build_text_blob_from_dict(group)
            group_type = normalize_text(group.get("Tipo", ""))

            perguntas = group.get("Perguntas", [])
            if not isinstance(perguntas, list):
                continue

            for pergunta in perguntas:
                if not isinstance(pergunta, dict):
                    continue

                var_code = normalize_var_code(pergunta.get("CodigoVariavel"))
                if not var_code:
                    continue

                append_unique(item["variables"], var_code)

                question_text = build_text_blob_from_dict(pergunta)
                vm_texts = candidate_texts_from_variable_map(var_code)

                context = {
                    "description": vm_texts.get("description", ""),
                    "reference": vm_texts.get("reference", ""),
                    "question_text": question_text,
                    "question_group_text": group_text,
                    "quadro_title": quadro_title,
                }

                section = classify_section(context)

                if section == "information" and group_type in ["produto", "listadeprodutos"]:
                    g_section = classify_section(
                        {
                            "description": "",
                            "reference": "",
                            "question_text": "",
                            "question_group_text": group_text,
                            "quadro_title": quadro_title,
                        }
                    )
                    if g_section != "information":
                        section = g_section

                append_unique(item["sections"][section], var_code)

        quadro_id = str(quadro_data.get("Id", "")).zfill(2)
        present = set(item["variables"])
        keep_adding = True
        while keep_adding:
            keep_adding = False
            for var_code, var_meta in variable_map.items():
                if not isinstance(var_meta, dict):
                    continue
                if not (
                    var_code.startswith(f"V{quadro_id}")
                    or var_code.startswith(f"VW{quadro_id}")
                    or var_code.startswith(f"VX{quadro_id}")
                ):
                    continue

                cond_refs = extract_var_codes_from_condition(var_meta.get("condition"))
                if cond_refs and (cond_refs & present) and var_code not in present:
                    append_unique(item["variables"], var_code)
                    present.add(var_code)
                    keep_adding = True

                    vm_texts = candidate_texts_from_variable_map(var_code)
                    ctx = {
                        "description": vm_texts.get("description", ""),
                        "reference": vm_texts.get("reference", ""),
                        "question_text": "",
                        "question_group_text": "",
                        "quadro_title": quadro_title,
                    }
                    sec = classify_section(ctx)
                    append_unique(item["sections"][sec], var_code)

        classified = set(
            item["sections"]["information"]
            + item["sections"]["area"]
            + item["sections"]["production"]
            + item["sections"]["sells"]
        )
        for var_code in item["variables"]:
            if var_code not in classified:
                append_unique(item["sections"][FALLBACK_SECTION], var_code)

        quad_map[quadro_name] = item

    return quad_map

candidate_form = build_candidate_form_structure()

if ENFORCE_REFERENCE_OUTPUT:
    # Força a estrutura de saída a seguir o formato de referência, mesmo que haja diferenças de conteúdo.
    final_output = OrderedDict()
    for quadro_name, ref_item in reference_form.items():
        item = {
            "title": ref_item.get("title", ""),
            "variables": ref_item.get("variables", []),
            "sections": ref_item.get("sections", {}),
        }
        if INCLUDE_SHOW_UNITS and "show_units" in ref_item:
            item["show_units"] = ref_item.get("show_units", [])
        final_output[quadro_name] = item
else:
    final_output = candidate_form

with open(_OUT_FORM, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4, ensure_ascii=False)

Diagnostico da classificacao textual: 39 quadros com divergencia
Arquivo gerado: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/generated/form_structure_generated.json
Validacao: OK. Saida identica ao form_structure.json
Total de quadros validados: 40
